# Evaluación: Comparar experimentos y decidir la arquitectura campeona

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from datetime import datetime
import os
import pandas as pd
from sklearn.metrics import f1_score

from src import models, utils

# Parámetros

In [3]:
VERSION_ID = '202602'  # model_version_id
SCORE = "F1"

# Paths
path_interim = os.path.join("data", "interim")
path_experiment1 =  os.path.join(path_interim, "exp01_hpt_nb")
path_experiment2 =  os.path.join(path_interim, "exp02_hpt_gbt")
path_model_prod = os.path.join("models", "prod")
path_model_arch = os.path.join("models", "archive")

# Input
file_train = "train.csv"
file_test = "test.csv"

In [4]:
if not os.path.exists(path_model_prod):
    print(f"Creating the folder: {path_model_prod}")
    os.mkdir(path_model_prod)
if not os.path.exists(path_model_arch):
    print(f"Creating the folder: {path_model_arch}")
    os.mkdir(path_model_arch)

# Cargar datos

## Subconjuntos Entrenamiento/Prueba

In [5]:
path_data_train = os.path.join(path_interim, file_train)

df_train = pd.read_csv(path_data_train)
df_train.head(2)

,x_text,y_is_nf
0,Respuestas coherentes e idénticas ante entrada...,0
1,Gestión de usuarios: Todos los administradores...,0


In [6]:
path_data_test = os.path.join(path_interim, file_test)

df_test = pd.read_csv(path_data_test)
df_test.head(2)

,x_text,y_is_nf
0,Como administrador quiero comparar el rendimie...,0
1,La herramienta debe permitir el aumento del nú...,1


## Resultados de experimentos

In [7]:
df_cv_summary_exp1 = pd.read_csv(
    os.path.join(path_experiment1,"df_exp_summary.csv")
)
df_cv_summary_exp1.head(2)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_fte__max_df,param_fte__max_features,param_fte__min_df,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,experiment_id
0,0.376776,0.108278,0.210442,0.039959,0.5,NaN,1,"{'fte__max_df': 0.5, 'fte__max_features': None...",0.529412,0.684211,0.615385,0.609669,0.063325,34,exp01_hpt_nb
1,0.336934,0.103631,0.180520,0.053305,0.5,NaN,3,"{'fte__max_df': 0.5, 'fte__max_features': None...",0.711111,0.769231,0.777778,0.752707,0.029619,19,exp01_hpt_nb


In [8]:
df_cv_summary_exp2 = pd.read_csv(
     os.path.join(path_experiment2,"df_exp_summary.csv")
)
df_cv_summary_exp2.head(2)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_clf__max_depth,param_fte__max_df,param_fte__max_features,param_fte__min_df,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,experiment_id
0,3.475810,0.102856,0.136961,0.009637,3,0.25,64,1,"{'clf__max_depth': 3, 'fte__max_df': 0.25, 'ft...",0.711111,0.800000,0.409091,0.640067,0.167308,45,exp02_hpt_gbt
1,3.720479,0.197966,0.158627,0.022464,3,0.25,64,3,"{'clf__max_depth': 3, 'fte__max_df': 0.25, 'ft...",0.711111,0.826087,0.409091,0.648763,0.175854,39,exp02_hpt_gbt


# Evaluación

In [9]:
df_cv_summary_exp1.sort_values(ascending=True, by="rank_test_score").head(5)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_fte__max_df,param_fte__max_features,param_fte__min_df,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,experiment_id
14,0.512938,0.008227,0.297304,0.007301,0.75,NaN,5,"{'fte__max_df': 0.75, 'fte__max_features': Non...",0.734694,0.769231,0.800000,0.767975,0.026676,1,exp01_hpt_nb
23,0.492036,0.005984,0.240886,0.007022,0.75,256.0,5,"{'fte__max_df': 0.75, 'fte__max_features': 256...",0.734694,0.769231,0.800000,0.767975,0.026676,1,exp01_hpt_nb
26,0.474908,0.002054,0.229641,0.005178,0.95,NaN,5,"{'fte__max_df': 0.95, 'fte__max_features': Non...",0.734694,0.769231,0.800000,0.767975,0.026676,1,exp01_hpt_nb
35,0.345629,0.033044,0.070221,0.007446,0.95,256.0,5,"{'fte__max_df': 0.95, 'fte__max_features': 256...",0.734694,0.769231,0.800000,0.767975,0.026676,1,exp01_hpt_nb
8,0.469502,0.006064,0.237005,0.004107,0.50,128.0,5,"{'fte__max_df': 0.5, 'fte__max_features': 128,...",0.750000,0.754717,0.792453,0.765723,0.018998,5,exp01_hpt_nb


In [10]:
df_cv_summary_exp2.sort_values(ascending=True, by="rank_test_score").head(5)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_clf__max_depth,param_fte__max_df,param_fte__max_features,param_fte__min_df,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,experiment_id
40,5.736805,0.277375,0.193339,0.059588,7,0.25,256,1,"{'clf__max_depth': 7, 'fte__max_df': 0.25, 'ft...",0.734694,0.723404,0.731707,0.729935,0.004776,1,exp02_hpt_gbt
46,5.644761,0.128629,0.310168,0.018758,7,0.50,256,1,"{'clf__max_depth': 7, 'fte__max_df': 0.5, 'fte...",0.734694,0.723404,0.731707,0.729935,0.004776,1,exp02_hpt_gbt
47,6.118877,0.720558,0.208212,0.044737,7,0.50,256,3,"{'clf__max_depth': 7, 'fte__max_df': 0.5, 'fte...",0.720000,0.695652,0.731707,0.715786,0.015018,3,exp02_hpt_gbt
41,5.813476,0.385993,0.207325,0.050984,7,0.25,256,3,"{'clf__max_depth': 7, 'fte__max_df': 0.25, 'ft...",0.720000,0.695652,0.731707,0.715786,0.015018,3,exp02_hpt_gbt
52,4.747097,0.471570,0.103352,0.011938,7,0.95,256,1,"{'clf__max_depth': 7, 'fte__max_df': 0.95, 'ft...",0.693878,0.723404,0.714286,0.710523,0.012344,5,exp02_hpt_gbt


# Modelo campeón

In [11]:
df_cv_summary_exp1.loc[
    df_cv_summary_exp1['rank_test_score'] == 1, [
        "mean_test_score", "std_test_score",
        "param_fte__max_df","param_fte__max_features",	"param_fte__min_df"]  # set the params of your champion model
]   # at a tie, you can get the model with : lowest std_test_score and the most simple one

,mean_test_score,std_test_score,param_fte__max_df,param_fte__max_features,param_fte__min_df
14,0.767975,0.026676,0.75,NaN,5
23,0.767975,0.026676,0.75,256.0,5
26,0.767975,0.026676,0.95,NaN,5
35,0.767975,0.026676,0.95,256.0,5


Ve a src/models.py e implementa get_model()


```python
def get_model(
    # your Hiperparameters:
    min_df: int = 3,
    max_df: float = 0.5,
    ...
):
    """
    Construye y retorna una Tubería de scikit-learn para clasificación de texto en español

    Args:
        min_df (int): Frecuencia mínima de documento para el vectorizador.
        max_df (float): Frecuencia máxima de documento para el vectorizador.
        max_features (int, optional): Número máximo de características a incluir.

    Returns:
        sklearn.pipeline.Pipeline: Una tubería con vectorizador y clasificador.
    """

    
    logging.info("Building pipeline...")

    # Tu estrategia de vectorización
    dtm_transformer = XXXVectorizer(
        ...
    )

    # Tu modelo
    clf = ...


    # Arquitectura de tubería campeona
    skl_pl = Pipeline([
        ('fte', dtm_transformer),
        ('clf', clf)
    ])

    return skl_pl
```